# Module 02 — Assignment: The Training Toolkit

Module 01's assignment had you derive backprop. Now you implement the three
pieces that turn "it learns" into "it learns fast": a **ReLU** layer, **He
initialization**, and the **Adam** update. Each is a load-bearing part of the
shared `nanograd` library — you fill the `# TODO` blocks, and the check cells
grade you against a finite-difference gradient check and the canonical
`python/toolkit.py` answer key.

Rules: don't peek at `lib/python/nanograd` or `python/toolkit.py` until you've
tried. The check cells are your feedback. Fill every `# TODO`, run top to bottom.

In [ ]:
# Setup — run once. (GIVEN: do not modify.)
import os, sys, math
import numpy as np

def _find_repo():
    here = os.getcwd()
    cands = [here] + [os.path.abspath(os.path.join(here, *(['..'] * k))) for k in range(1, 6)]
    for base in cands:
        if os.path.isdir(os.path.join(base, 'lib', 'python', 'nanograd')):
            return base
    return here

REPO = _find_repo()
TOPIC = os.path.join(REPO, 'topics', '02-training-toolkit')
sys.path.insert(0, os.path.join(REPO, 'lib', 'python'))
sys.path.insert(0, os.path.join(TOPIC, 'python'))
sys.path.insert(0, os.path.join(TOPIC, 'tests'))
import nanograd as ng
import toolkit as canon
from check_utils import (rel_error, eval_numerical_gradient_array, compare_to_canonical)
from nanograd.rng import Rng
print('setup ok — nanograd and check utils imported')

## Part 1 — the ReLU layer

$\mathrm{ReLU}(z)=\max(0,z)$. Forward, it zeroes negatives. Backward, the
gradient passes **only where the unit was active** ($z>0$) and is blocked
everywhere else — that on/off gate is the whole point. Implement both.

In [ ]:
class MyReLU:
    def forward(self, Z):
        mask, out = None, None
    ###########################################################################
    # TODO: cache the boolean mask (Z > 0) as self.mask, and return Z with negatives
    # zeroed. Two lines.
    ###########################################################################
        # mask = ...
        # out = ...
    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
        self.mask = mask
        return out

    def backward(self, dA):
        dZ = None
    ###########################################################################
    # TODO: pass the incoming gradient dA through only where the unit was active
    # (use self.mask). One line.
    ###########################################################################
        # dZ = ...
    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
        return dZ

In [ ]:
# Check: gradient-check your backward, and match nanograd's ReLU exactly.
Z = np.random.default_rng(0).standard_normal((5, 5))
dA = np.random.default_rng(1).standard_normal((5, 5))
layer = MyReLU(); layer.forward(Z); analytic = layer.backward(dA)
numeric = eval_numerical_gradient_array(lambda z: MyReLU().forward(z), Z, dA)
print(f'ReLU backward vs finite differences: rel_error = {rel_error(analytic, numeric):.2e}')
assert rel_error(analytic, numeric) < 1e-6
ref = ng.ReLU(); ref.forward(Z)
compare_to_canonical([MyReLU().forward(Z).sum(), analytic.sum()],
                     [ref.forward(Z).sum(), (ref.backward(dA)).sum()],
                     labels=['fwd_sum', 'bwd_sum'])

### Inline Question 1
A sigmoid's derivative is at most `0.25` and decays to ~0 for large `|z|`; ReLU's
is exactly `1` for all `z > 0`. In a *deep* stack, why does that difference keep
gradients from vanishing? *Your answer:* ____

## Part 2 — He initialization

For a ReLU layer with `fan_in` inputs, draw weights from a Gaussian with
$\mathrm{std}=\sqrt{2/\text{fan\_in}}$. The factor of 2 compensates for ReLU
zeroing half its inputs, keeping the activation variance stable across depth.
Fill `W` **in place**, drawing row-major from `rng.normal()` so it matches the
library bit-for-bit.

In [ ]:
def he_init(W, rng):
    fan_in, fan_out = W.shape
    ###########################################################################
    # TODO: set std = sqrt(2 / fan_in), then fill W in row-major order (i outer, o inner)
    # with std * rng.normal().
    ###########################################################################
    # std = ...
    # for i in range(fan_in):
    #     for o in range(fan_out):
    #         W[i, o] = ...
    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
    return W

In [ ]:
# Check: your He fill must match nanograd.he_normal drawing from the same seed.
Wa = np.zeros((8, 5)); he_init(Wa, Rng(42))
Wb = np.zeros((8, 5)); ng.he_normal(Wb, Rng(42))
print(f'std(you)={Wa.std():.4f}   std(nanograd)={Wb.std():.4f}   target={math.sqrt(2/8):.4f}')
compare_to_canonical([Wa[0, 0], Wa[3, 2], float(Wa.sum())],
                     [Wb[0, 0], Wb[3, 2], float(Wb.sum())],
                     labels=['W[0,0]', 'W[3,2]', 'sum'])

### Inline Question 2
If you used Xavier ($\sqrt{1/\text{fan\_in}}$) instead of He for a deep ReLU
net, would the activations tend to grow or shrink with depth, and why? *Your
answer:* ____

## Part 3 — the Adam update

Adam keeps a running mean `m` (momentum) and mean-square `v` (RMSProp) of the
gradient, bias-corrects both for their cold start at zero, and takes a
per-parameter step:

$$m \leftarrow \beta_1 m + (1{-}\beta_1) g,\quad v \leftarrow \beta_2 v + (1{-}\beta_2) g^2$$
$$\hat m = m/(1-\beta_1^{\,t}),\ \hat v = v/(1-\beta_2^{\,t}),\qquad
\theta \mathrel{-}= \eta\,\hat m/(\sqrt{\hat v}+\epsilon)$$

Implement one step over the arrays `p, g, m, v` (all same shape), updating them
**in place**. `t` is the 1-based step number.

In [ ]:
def adam_step(p, g, m, v, t, lr=1e-2, b1=0.9, b2=0.999, eps=1e-8):
    ###########################################################################
    # TODO: update m and v in place (elementwise), bias-correct with t, then update p
    # in place. Use m[:] = ... and v[:] = ... so the arrays mutate.
    ###########################################################################
    # m[:] = ...
    # v[:] = ...
    # m_hat = ...
    # v_hat = ...
    # p -= ...
    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
    return p

In [ ]:
# Check: 20 steps of your Adam vs nanograd.Adam from identical p0 and gradients.
rng = np.random.default_rng(7)
p0 = rng.standard_normal((4, 3)); g = rng.standard_normal((4, 3))
p_you = p0.copy(); m = np.zeros_like(p0); v = np.zeros_like(p0)
for t in range(1, 21):
    adam_step(p_you, g, m, v, t)
p_ref = p0.copy(); opt = ng.Adam(lr=1e-2)
for _ in range(20):
    opt.step([p_ref], [g])
print(f'Adam trajectory rel_error = {rel_error(p_you, p_ref):.2e}')
compare_to_canonical([float(p_you.sum()), float(p_you[0, 0])],
                     [float(p_ref.sum()), float(p_ref[0, 0])], labels=['sum', 'p[0,0]'])

### Inline Question 3
Early in training, `t` is small so `1 - β₂ᵗ` is tiny and `v̂ = v/(1-β₂ᵗ)` divides
by a small number. What goes wrong if you **drop** the bias-correction and use
`v` directly in the first few steps? *Your answer:* ____

## Verify: your pieces compose to train the net

Each part above already graded a single piece against the canonical `nanograd`
library to `1e-9`. This last cell is the composition test: your ReLU, He init,
and Adam are dropped into a GIVEN training harness (the same `2→8→2` net as
`python/toolkit.py`) and must drive XOR to **100% accuracy** with a near-zero
loss. (We don't compare final *weights* to the C/scalar answer key here — many
different weight settings solve XOR, and the NumPy and scalar paths converge to
different ones; the per-piece `1e-9` checks above are the exact gates.)

In [ ]:
def train_toy_with(relu_cls, init_fn, step_fn, steps=600):
    '''GIVEN harness: trains the toy net using YOUR relu/init/adam. Do not modify.'''
    rng = Rng(canon.TOY_SEED)
    L1, L2 = ng.Linear(2, 8), ng.Linear(8, 2)
    init_fn(L1.W, rng); L1.b[:] = 0.0
    init_fn(L2.W, rng); L2.b[:] = 0.0
    act, lossf = relu_cls(), ng.SoftmaxCrossEntropy()
    X = np.array(canon.TOY_X); y = np.array(canon.TOY_Y)
    state = {id(P): [np.zeros_like(P), np.zeros_like(P)] for P in (L1.W, L1.b, L2.W, L2.b)}
    for t in range(1, steps + 1):
        H = act.forward(L1.forward(X)); lossf.forward(L2.forward(H), y)
        dH = L2.backward(lossf.backward()); L1.backward(act.backward(dH))
        for P, G in ((L1.W, L1.dW), (L1.b, L1.db), (L2.W, L2.dW), (L2.b, L2.db)):
            m, v = state[id(P)]; step_fn(P, G, m, v, t, lr=0.05)
    logits = L2.forward(act.forward(L1.forward(X)))   # fresh forward, post-update
    lossf.forward(logits, y)                           # populates lossf.P
    loss = float(-np.log(lossf.P[np.arange(4), y] + 1e-12).mean())
    acc = float((logits.argmax(1) == y).mean())
    wsum = float(L1.W.sum() + L2.W.sum() + L1.b.sum() + L2.b.sum())
    return loss, acc, wsum

loss, acc, wsum = train_toy_with(MyReLU, he_init, adam_step)
print(f'trained XOR with YOUR pieces:  acc = {acc:.3f}   loss = {loss:.6f}')
assert acc == 1.0, 'your pieces did not reach 100% on XOR — check parts 1-3'
assert loss < 1e-3, f'XOR trained but loss {loss:.4g} is too high — check the Adam step'
print('\nDone — your ReLU, He init, and Adam compose into a working trainer.')